# Modelling dataset: build and quality checks
### Hidden Dissatisfaction · Olist Brazilian e-commerce dataset

`01_eda_why_lateness_dominates` explains *why* the model is restricted to on-time orders. This notebook builds that table and checks it before any model is fitted. The general audit of the nine raw files is in `olist_open_eda` (§1–§2); the checks here are the ones specific to the modelling table.

| Item | Definition |
|---|---|
| Grain | one row per order (`order_id`) |
| Population | delivered, reviewed, bought Jan 2017 – Aug 2018, delivered **on time** |
| Prediction point | the moment the order is delivered |
| Target | `negative` = `review_score` of 1 or 2 stars |
| Split | by purchase date: train Jan–Dec 2017, validation Jan–Mar 2018, test Apr–Aug 2018 |

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import ks_2samp
from sklearn.metrics import roc_auc_score

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.max_rows", 100)
DATA = Path("dataset")  # folder with the 9 Olist CSV files
WINDOW = (pd.Timestamp("2017-01-01"), pd.Timestamp("2018-09-01"))  # usable purchase months
VAL_START, TEST_START = pd.Timestamp("2018-01-01"), pd.Timestamp("2018-04-01")  # train | validation | test


def show(df, pct=(), num=(), dec=3):
    """Display a table with readable number formats (no Styler/jinja2 needed)."""
    out = df.copy()
    for c in pct:
        out[c] = out[c].map(lambda v: "" if pd.isna(v) else f"{v:.1%}")
    for c in num:
        out[c] = out[c].map(lambda v: "" if pd.isna(v) else (f"{v:,.0f}" if float(v).is_integer() else f"{v:,.{dec}f}"))
    display(out)


def auc_free(y, x):
    """Single-feature ROC-AUC with direction ignored (0.5 = no signal)."""
    ok = x.notna()
    if x[ok].nunique() < 2:
        return np.nan
    a = roc_auc_score(y[ok].astype(int), x[ok].astype(float))
    return max(a, 1 - a)

## 1 · Build one row per order, with a grain check after every join

Same logic as `01` §1, with these differences:

- every merge declares its expected cardinality (`validate=`), and `order_id` uniqueness is asserted after each step;
- zip centroids drop coordinates outside Brazil and use the median point instead of the mean;
- a category without an English translation keeps its Portuguese name; a product without a category becomes `(missing)`;
- ties for the most expensive item are broken by `order_item_id`, so the main seller and category are deterministic;
- orders with several reviews, or a review shared with other orders, are flagged instead of silently resolved.

In [2]:
orders = pd.read_csv(DATA / "olist_orders_dataset.csv", parse_dates=[
    "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
    "order_delivered_customer_date", "order_estimated_delivery_date"])
reviews = pd.read_csv(DATA / "olist_order_reviews_dataset.csv", parse_dates=["review_creation_date", "review_answer_timestamp"])
items = pd.read_csv(DATA / "olist_order_items_dataset.csv", parse_dates=["shipping_limit_date"])
payments = pd.read_csv(DATA / "olist_order_payments_dataset.csv")
products = pd.read_csv(DATA / "olist_products_dataset.csv")
customers = pd.read_csv(DATA / "olist_customers_dataset.csv", dtype={"customer_zip_code_prefix": str})
sellers = pd.read_csv(DATA / "olist_sellers_dataset.csv", dtype={"seller_zip_code_prefix": str})
geo = pd.read_csv(DATA / "olist_geolocation_dataset.csv", dtype={"geolocation_zip_code_prefix": str})
cat_en = pd.read_csv(DATA / "product_category_name_translation.csv")

# 1) one review per order (most recent answer), keeping flags for the ambiguous cases
per_order = reviews.groupby("order_id").review_score.agg(n_reviews="size", n_scores="nunique").reset_index()
shared_orders = reviews.loc[reviews.review_id.duplicated(keep=False), "order_id"].unique()
reviews_1 = (reviews.sort_values("review_answer_timestamp").drop_duplicates("order_id", keep="last")
             .merge(per_order, on="order_id", validate="one_to_one"))
reviews_1["review_conflict"] = reviews_1.n_scores > 1               # several reviews with different scores
reviews_1["review_id_shared"] = reviews_1.order_id.isin(shared_orders)  # one review attached to several orders

# 2) items -> order level (the most expensive item defines the order's category and main seller)
products["category"] = (products.product_category_name
                        .map(cat_en.set_index("product_category_name").product_category_name_english)
                        .fillna(products.product_category_name).fillna("(missing)"))
it = items.merge(products, on="product_id", how="left", validate="many_to_one")
order_items = it.groupby("order_id").agg(
    n_items=("order_item_id", "size"), n_products=("product_id", "nunique"), n_sellers=("seller_id", "nunique"),
    price=("price", "sum"), freight=("freight_value", "sum"), shipping_limit=("shipping_limit_date", "max"),
    photos=("product_photos_qty", "mean"), desc_len=("product_description_lenght", "mean"),
    weight_g=("product_weight_g", "sum")).reset_index()
main_item = (it.sort_values(["price", "order_item_id"], ascending=[False, True]).drop_duplicates("order_id")
             [["order_id", "category", "seller_id"]].rename(columns={"seller_id": "main_seller"}))

# 3) payments -> order level
order_pay = payments.groupby("order_id").agg(n_pay=("payment_sequential", "size"), installments=("payment_installments", "max"),
                                             paid=("payment_value", "sum")).reset_index()
main_pay = (payments.sort_values(["payment_value", "payment_sequential"], ascending=[False, True])
            .drop_duplicates("order_id")[["order_id", "payment_type"]])

# 4) join everything; each step must keep one row per order
review_cols = ["order_id", "review_score", "review_comment_message", "review_creation_date", "review_answer_timestamp",
               "n_reviews", "review_conflict", "review_id_shared"]
steps = [
    ("reviews (inner: reviewed orders only)", reviews_1[review_cols], "order_id", "inner"),
    ("items, aggregated per order", order_items, "order_id", "left"),
    ("main item (category, main seller)", main_item, "order_id", "left"),
    ("payments, aggregated per order", order_pay, "order_id", "left"),
    ("main payment type", main_pay, "order_id", "left"),
    ("customer", customers[["customer_id", "customer_unique_id", "customer_zip_code_prefix", "customer_state"]],
     "customer_id", "left"),
    ("main seller", sellers.rename(columns={"seller_id": "main_seller"}), "main_seller", "left"),
]
df, log = orders, [("orders (start)", len(orders), 0)]
for name, right, key, how in steps:
    df = df.merge(right, on=key, how=how, validate="many_to_one")
    assert df.order_id.is_unique, f"fan-out after joining {name}"
    log.append((name, len(df), int((~df[key].isin(right[key])).sum())))
show(pd.DataFrame(log, columns=["join", "rows after join", "rows without a match"]), num=["rows after join", "rows without a match"])

,join,rows after join,rows without a match
0,orders (start),"99,441",0
1,reviews (inner: reviewed orders only),"98,673",0
2,"items, aggregated per order","98,673",756
3,"main item (category, main seller)","98,673",756
4,"payments, aggregated per order","98,673",1
5,main payment type,"98,673",1
6,customer,"98,673",0
7,main seller,"98,673",756


**What we see.** No join changes the row count: 98,673 reviewed orders before and after every left join. 756 reviewed orders have no item rows (and therefore no main seller or category) and 1 has no payment row. All of the item-less orders are undelivered, so none reaches the modelling table (§2 confirms 0).

## 2 · Features, and values that cannot be true

Timing features are derived as in `01`. Before they are used, each rule below counts the delivered-and-reviewed orders that break it and states what is done about them.

In [3]:
# seller <-> customer distance from cleaned zip-prefix centroids (haversine, km)
geo_ok = geo[geo.geolocation_lat.between(-33.8, 5.3) & geo.geolocation_lng.between(-74.0, -34.7)]
centroid = geo_ok.groupby("geolocation_zip_code_prefix")[["geolocation_lat", "geolocation_lng"]].median()
c = centroid.reindex(df.customer_zip_code_prefix.to_numpy()).to_numpy()
s = centroid.reindex(df.seller_zip_code_prefix.to_numpy()).to_numpy()
lat1, lng1, lat2, lng2 = map(np.radians, (c[:, 0], c[:, 1], s[:, 0], s[:, 1]))
h = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lng2 - lng1) / 2) ** 2
df["distance_km"] = 6371 * 2 * np.arcsin(np.sqrt(h))
df["negative"] = df.review_score <= 2

# delivered orders with a delivery date, then the timing features
d = df[(df.order_status == "delivered") & df.order_delivered_customer_date.notna()].copy()
day = pd.Timedelta(days=1)
d["days_vs_estimate"] = (d.order_delivered_customer_date.dt.normalize() - d.order_estimated_delivery_date).dt.days
d["is_late"] = d.days_vs_estimate > 0
d["delivery_days"] = (d.order_delivered_customer_date - d.order_purchase_timestamp) / day
d["promised_days"] = (d.order_estimated_delivery_date - d.order_purchase_timestamp) / day
d["approval_hours"] = (d.order_approved_at - d.order_purchase_timestamp) / pd.Timedelta(hours=1)
d["handoff_days"] = (d.order_delivered_carrier_date - d.order_purchase_timestamp) / day
d["carrier_to_customer_days"] = (d.order_delivered_customer_date - d.order_delivered_carrier_date) / day
d["dispatch_late"] = d.order_delivered_carrier_date > d.shipping_limit
d["freight_ratio"] = d.freight / d.price
# review-process fields: used ONLY to check the label, never as model features
d["answered_before_purchase"] = d.review_answer_timestamp < d.order_purchase_timestamp
d["survey_sent_before_delivery"] = d.review_creation_date < d.order_delivered_customer_date.dt.normalize()
d["answered_before_delivery"] = d.review_answer_timestamp < d.order_delivered_customer_date

rules = [
    ("Carrier pick-up before purchase", d.handoff_days < 0, "handoff_days set to missing"),
    ("Delivered to customer before carrier pick-up", d.carrier_to_customer_days < 0, "carrier_to_customer_days set to missing"),
    ("Carrier pick-up before payment approval", d.order_delivered_carrier_date < d.order_approved_at, "kept; no feature measures this gap"),
    ("Carrier pick-up date missing", d.order_delivered_carrier_date.isna(), "hand-off features stay missing"),
    ("Payment approval date missing", d.order_approved_at.isna(), "approval_hours stays missing"),
    ("Payment approved before purchase", d.approval_hours < 0, "none expected"),
    ("Delivered before purchase", d.delivery_days < 0, "none expected"),
    ("Estimated date before purchase", d.promised_days < 0, "none expected"),
    ("Review answered before purchase", d.answered_before_purchase, "label flagged uncertain; left out of seller history"),
    ("No item rows", d.n_items.isna(), "none expected for delivered orders"),
    ("No payment rows", d.n_pay.isna(), "payment features stay missing"),
    ("Instalments = 0", d.installments == 0, "set to 1"),
    ("Payment type 'not_defined'", d.payment_type == "not_defined", "none expected for delivered orders"),
    ("Order weight = 0 g", d.weight_g == 0, "weight_g set to missing"),
    ("Order price = 0", d.price <= 0, "none expected"),
    ("Freight = 0", d.freight == 0, "kept (free shipping)"),
    ("Items + freight differs from amount paid by > 1 BRL", (d.price + d.freight - d.paid).abs() > 1, "kept; amount paid is not a feature"),
    ("Customer zip prefix has no centroid", centroid.reindex(d.customer_zip_code_prefix.to_numpy()).geolocation_lat.isna().to_numpy(), "distance_km stays missing"),
    ("Seller zip prefix has no centroid", centroid.reindex(d.seller_zip_code_prefix.to_numpy()).geolocation_lat.isna().to_numpy(), "distance_km stays missing"),
]
print(f"Delivered and reviewed orders: {len(d):,} | geolocation rows outside Brazil dropped: {len(geo) - len(geo_ok):,}")
show(pd.DataFrame([(name, int(np.sum(mask)), np.mean(mask), fix) for name, mask, fix in rules],
                  columns=["rule", "orders", "share", "handling"]), num=["orders"], pct=["share"])

d.loc[d.handoff_days < 0, "handoff_days"] = np.nan
d.loc[d.carrier_to_customer_days < 0, "carrier_to_customer_days"] = np.nan
d["installments"] = d.installments.replace(0, 1)
d["weight_g"] = d.weight_g.replace(0, np.nan)

Delivered and reviewed orders: 95,824 | geolocation rows outside Brazil dropped: 42


,rule,orders,share,handling
0,Carrier pick-up before purchase,164,0.2%,handoff_days set to missing
1,Delivered to customer before carrier pick-up,23,0.0%,carrier_to_customer_days set to missing
2,Carrier pick-up before payment approval,"1,343",1.4%,kept; no feature measures this gap
3,Carrier pick-up date missing,1,0.0%,hand-off features stay missing
4,Payment approval date missing,14,0.0%,approval_hours stays missing
5,Payment approved before purchase,0,0.0%,none expected
6,Delivered before purchase,0,0.0%,none expected
7,Estimated date before purchase,0,0.0%,none expected
8,Review answered before purchase,3,0.0%,label flagged uncertain; left out of seller history
9,No item rows,0,0.0%,none expected for delivered orders


**What we see.** The delivered-and-reviewed table has **95,824 orders, the same as `01`**, so the stricter build loses nothing.
- **Impossible timings are rare:** 164 orders were picked up by the carrier before they were bought and 23 reached the customer before the carrier had them. Those two durations are set to missing instead of entering the model as negative numbers.
- **1,343 orders (1.4%)** were picked up before the payment was approved. Both dates are kept as they are; no feature measures the gap between them.
- **Distance is missing for about 0.5% of orders** (264 customer and 213 seller zip prefixes without a centroid). 42 geolocation rows outside Brazil were dropped before the centroids were computed.
- 22 orders weigh 0 g and 2 have 0 instalments; both are corrected. 335 orders have free shipping and 244 show a paid amount that differs from items + freight by more than 1 BRL; both are left alone.

## 3 · Point-in-time seller history

For each order, the seller's reviews **answered before the purchase** are counted (at least 5 are needed for a rate). `01` built this history from delivered orders only. A seller's undelivered orders are also reviewed, mostly badly, and those reviews are just as visible at purchase time, so they are included here. A review answered before its own order's purchase is excluded, otherwise an order could see its own label.

In [4]:
def seller_history(source):
    """Per order in `d`: the sellers' earlier answered reviews (smallest count, highest 1-2 star share)."""
    src = source[source.review_answer_timestamp > source.order_purchase_timestamp]
    pairs = items[["order_id", "seller_id"]].drop_duplicates()
    events = (pairs.merge(src[["order_id", "review_answer_timestamp", "negative"]], on="order_id")
              .sort_values("review_answer_timestamp"))
    events["prior_n"] = events.groupby("seller_id").cumcount() + 1
    events["prior_neg"] = events.groupby("seller_id").negative.cumsum()
    queries = pairs.merge(d[["order_id", "order_purchase_timestamp"]], on="order_id").sort_values("order_purchase_timestamp")
    hist = pd.merge_asof(queries, events[["seller_id", "review_answer_timestamp", "prior_n", "prior_neg"]],
                         left_on="order_purchase_timestamp", right_on="review_answer_timestamp", by="seller_id",
                         allow_exact_matches=False)
    hist[["prior_n", "prior_neg"]] = hist[["prior_n", "prior_neg"]].fillna(0)
    hist["rate"] = np.where(hist.prior_n >= 5, hist.prior_neg / hist.prior_n.clip(lower=1), np.nan)
    return hist.groupby("order_id").agg(seller_prior_n=("prior_n", "min"), seller_prior_neg_rate=("rate", "max"))


rows_before = len(d)
d = d.join(seller_history(df), on="order_id")                                      # all reviewed orders
d = d.join(seller_history(d).add_suffix("_delivered_only"), on="order_id")         # the version used in 01
assert len(d) == rows_before and d.order_id.is_unique

in_scope = d[d.order_purchase_timestamp.between(*WINDOW, inclusive="left") & ~d.is_late]
undelivered = df[~df.order_id.isin(d.order_id)]
print(f"Reviewed orders that were not delivered: {len(undelivered):,}, of which {undelivered.negative.mean():.1%} are 1-2 stars")
show(pd.DataFrame({
    "history from": ["all reviewed orders (used)", "delivered orders only (as in 01)"],
    "orders with a rate": [in_scope.seller_prior_neg_rate.notna().mean(), in_scope.seller_prior_neg_rate_delivered_only.notna().mean()],
    "mean rate": [in_scope.seller_prior_neg_rate.mean(), in_scope.seller_prior_neg_rate_delivered_only.mean()],
    "single-feature AUC": [auc_free(in_scope.negative, in_scope.seller_prior_neg_rate),
                           auc_free(in_scope.negative, in_scope.seller_prior_neg_rate_delivered_only)],
}), pct=["orders with a rate", "mean rate"], num=["single-feature AUC"])
d = d.drop(columns=["seller_prior_n_delivered_only", "seller_prior_neg_rate_delivered_only"])

Reviewed orders that were not delivered: 2,849, of which 78.0% are 1-2 stars


,history from,orders with a rate,mean rate,single-feature AUC
0,all reviewed orders (used),84.6%,13.3%,0.587
1,delivered orders only (as in 01),84.4%,12.4%,0.586


**What we see.** 2,849 reviewed orders were never delivered and 78% of them are 1–2★. Adding them raises the average seller rate from 12.4% to 13.3%, but the feature's ranking power on on-time orders barely moves (AUC 0.586 → 0.587). The change is kept because it is the correct point-in-time definition, not because it improves the score.

## 4 · Population: time window and on-time filter

The first table shows, per purchase month, how many orders exist and how many reach the delivered-and-reviewed table. The second is the funnel down to the modelling population.

In [5]:
cov = (orders.assign(month=orders.order_purchase_timestamp.dt.to_period("M").astype(str),
                     delivered=(orders.order_status == "delivered") & orders.order_delivered_customer_date.notna(),
                     in_table=orders.order_id.isin(d.order_id))
       .groupby("month").agg(orders=("order_id", "size"), delivered=("delivered", "mean"), delivered_and_reviewed=("in_table", "mean")))
show(cov.reset_index(), num=["orders"], pct=["delivered", "delivered_and_reviewed"])

in_window = d.order_purchase_timestamp.between(*WINDOW, inclusive="left")
m = d[in_window & ~d.is_late].copy()
assert m.order_id.is_unique
show(pd.DataFrame({"orders": {
    "All orders": len(orders),
    "Reviewed": len(df),
    "Delivered and reviewed": len(d),
    "…bought Jan 2017 – Aug 2018": int(in_window.sum()),
    "…and delivered on time = modelling table": len(m)}}), num=["orders"])
print(f"Negative (1-2 star) rate in the modelling table: {m.negative.mean():.1%}")

,month,orders,delivered,delivered_and_reviewed
0,2016-09,4,25.0%,25.0%
1,2016-10,324,81.8%,80.9%
2,2016-12,1,100.0%,100.0%
3,2017-01,800,93.8%,92.6%
4,2017-02,"1,780",92.9%,92.3%
5,2017-03,"2,682",94.9%,94.2%
6,2017-04,"2,404",95.8%,95.3%
7,2017-05,"3,700",95.8%,95.1%
8,2017-06,"3,245",96.6%,95.9%
9,2017-07,"4,026",96.2%,95.4%


,orders
All orders,"99,441"
Reviewed,"98,673"
Delivered and reviewed,"95,824"
…bought Jan 2017 – Aug 2018,"95,560"
…and delivered on time = modelling table,"89,182"


Negative (1-2 star) rate in the modelling table: 9.2%


**What we see.**
- **2016 and Sep–Oct 2018 are unusable:** 329 orders in 2016 with a gap in November, and 20 orders after August 2018, none delivered. They are dropped (264 delivered-and-reviewed orders).
- **No sign of right-censoring inside the window:** the delivered share in Jun–Aug 2018 (97.5–98.8%) is as high as in earlier months, so the last months are not missing their slow deliveries.
- **The modelling table has 89,182 on-time orders, 9.2% negative.**

## 5 · Label checks

The label is trustworthy only if it is one clear score given after the customer had the parcel. The table counts the on-time orders where that is in doubt.

In [6]:
m["label_uncertain"] = m.review_conflict | m.answered_before_purchase | m.answered_before_delivery
flags = {
    "Several reviews with different scores": m.review_conflict,
    "Several reviews, same score": (m.n_reviews > 1) & ~m.review_conflict,
    "Review shared with other orders (same review_id)": m.review_id_shared,
    "Review answered before purchase": m.answered_before_purchase,
    "Survey sent before delivery": m.survey_sent_before_delivery,
    "Review answered before delivery": m.answered_before_delivery,
    "label_uncertain (conflict, or answered before purchase/delivery)": m.label_uncertain,
}
show(pd.DataFrame({"orders": {k: int(v.sum()) for k, v in flags.items()},
                   "share of table": {k: v.mean() for k, v in flags.items()},
                   "negative rate in group": {k: m.negative[v].mean() for k, v in flags.items()}}).reset_index(names="case"),
     num=["orders"], pct=["share of table", "negative rate in group"])
print(f"Negative rate: all rows {m.negative.mean():.2%} | without label_uncertain {m.negative[~m.label_uncertain].mean():.2%}")

,case,orders,share of table,negative rate in group
0,Several reviews with different scores,176,0.2%,27.3%
1,"Several reviews, same score",319,0.4%,4.4%
2,Review shared with other orders (same review_id),"1,214",1.4%,16.9%
3,Review answered before purchase,3,0.0%,66.7%
4,Survey sent before delivery,221,0.2%,17.6%
5,Review answered before delivery,177,0.2%,16.9%
6,"label_uncertain (conflict, or answered before purchase/delivery)",352,0.4%,22.2%


Negative rate: all rows 9.25% | without label_uncertain 9.20%


**What we see.**
- **176 orders have several reviews with different scores.** The most recent one is kept, and 27% of these orders end up negative, three times the table average. Their label depends on an arbitrary choice.
- **177 orders were reviewed before the parcel arrived** although it was on time, so the score cannot reflect what was received.
- **1,214 orders (1.4%) share a review with other orders**, 17% negative. These are multi-order carts rated once; the score is real but it is not clear which order caused it. They are flagged separately (`review_id_shared`) and not counted as uncertain.
- `label_uncertain` covers 352 orders (0.4%). Removing them moves the negative rate from 9.25% to 9.20%.

**Why it matters.** The uncertain labels are too few to change the target, so the rows are kept and flagged. The modelling notebook can drop them from training or check that results hold without them.

## 6 · Missingness by feature

Share missing per model feature, and the negative rate when the value is missing against when it is present. A large gap means the missingness itself carries signal and should not be imputed away silently.

In [7]:
NUM = ["delivery_days", "days_vs_estimate", "promised_days", "handoff_days", "carrier_to_customer_days", "approval_hours",
       "n_items", "n_products", "n_sellers", "price", "freight", "freight_ratio", "distance_km", "installments", "n_pay",
       "photos", "desc_len", "weight_g", "seller_prior_neg_rate", "seller_prior_n"]
BIN, CAT = ["dispatch_late"], ["category", "customer_state", "payment_type"]   # is_late is constant in this table
FEATURES = NUM + BIN + CAT

miss = m[FEATURES].isna().assign(category=m.category.eq("(missing)"))
na = pd.DataFrame({
    "missing": miss.sum(), "share": miss.mean(),
    "negative rate if missing": [m.negative[miss[f]].mean() if miss[f].any() else np.nan for f in FEATURES],
    "negative rate if present": [m.negative[~miss[f]].mean() for f in FEATURES]})
show(na[na.missing > 0].sort_values("missing", ascending=False).reset_index(names="feature"),
     num=["missing"], pct=["share", "negative rate if missing", "negative rate if present"])
print("Features with no missing values:", ", ".join(na.index[na.missing == 0]))

,feature,missing,share,negative rate if missing,negative rate if present
0,seller_prior_neg_rate,"13,753",15.4%,8.5%,9.4%
1,category,"1,251",1.4%,12.5%,9.2%
2,photos,"1,228",1.4%,12.1%,9.2%
3,desc_len,"1,228",1.4%,12.1%,9.2%
4,distance_km,431,0.5%,7.0%,9.3%
5,handoff_days,162,0.2%,7.4%,9.3%
6,weight_g,22,0.0%,13.6%,9.2%
7,carrier_to_customer_days,19,0.0%,52.6%,9.2%
8,approval_hours,14,0.0%,7.1%,9.2%


Features with no missing values: delivery_days, days_vs_estimate, promised_days, n_items, n_products, n_sellers, price, freight, freight_ratio, installments, n_pay, seller_prior_n, dispatch_late, customer_state, payment_type


**What we see.**
- **Seller history is the only feature with substantial missingness (15.4%)**: sellers with fewer than 5 earlier answered reviews. Their negative rate (8.5%) is slightly below the rest (9.4%), so "no history" is not a risk signal by itself. `seller_prior_n` is always present and tells the model which case it is in.
- **Product information is missing for 1.4% of orders** (no category, photos or description), and those orders are more often negative (12.1–12.5% against 9.2%). The category level `(missing)` keeps that signal.
- Everything else is missing for under 0.5% of orders. The 52.6% negative rate for missing `carrier_to_customer_days` rests on 19 orders.

**Why it matters.** No feature needs to be dropped for missingness. A model that handles missing values natively (gradient boosting) can use the table as it is; a linear model needs imputation plus a missing-indicator for seller history and product information.

## 7 · Distributions and redundant features

Range and skew of each numeric feature, the size of each categorical feature, and the feature pairs that move together (Spearman, on-time orders only).

In [8]:
desc = m[NUM].describe(percentiles=[0.01, 0.5, 0.99]).T[["min", "1%", "50%", "99%", "max"]]
desc["skew"] = m[NUM].skew()
desc["distinct"] = m[NUM].nunique()
show(desc.reset_index(names="feature"), num=["min", "1%", "50%", "99%", "max", "skew", "distinct"], dec=2)

show(pd.DataFrame({f: {"levels": m[f].nunique(), "top level": m[f].value_counts().index[0],
                       "top level share": m[f].value_counts(normalize=True).iloc[0],
                       "levels with < 30 orders": int((m[f].value_counts() < 30).sum())} for f in CAT + BIN}).T
     .reset_index(names="feature"), pct=["top level share"])

rho = m[NUM + BIN].astype(float).corr(method="spearman")
pairs = rho.where(np.triu(np.ones(rho.shape, bool), k=1)).stack().rename_axis(["feature a", "feature b"]).rename("spearman").reset_index()
show(pairs[pairs.spearman.abs() >= 0.6].sort_values("spearman", key=abs, ascending=False).reset_index(drop=True), num=["spearman"], dec=2)

,feature,min,1%,50%,99%,max,skew,distinct
0,delivery_days,0.53,1.79,9.79,30.08,63.74,1.12,"86,622"
1,days_vs_estimate,-147,-35,-13,0,0,-1.18,82
2,promised_days,2.01,6.31,23.25,48.49,155.14,0.86,"86,973"
3,handoff_days,0.00,0.10,2.14,14.20,47.51,3.15,"80,925"
4,carrier_to_customer_days,0,0.81,6.95,25.63,178.27,1.51,"85,192"
5,approval_hours,0,0,0.34,89.64,319.53,3.44,"30,321"
6,n_items,1,1,1,3,20,6.93,16
7,n_products,1,1,1,2,8,8.00,8
8,n_sellers,1,1,1,2,5,9.61,5
9,price,0.85,11.99,85.03,987.27,"13,440",10.13,"7,383"


,feature,levels,top level,top level share,levels with < 30 orders
0,category,74,bed_bath_table,9.4%,11
1,customer_state,27,SP,43.0%,0
2,payment_type,4,credit_card,75.5%,0
3,dispatch_late,2,False,92.5%,0


,feature a,feature b,spearman
0,delivery_days,carrier_to_customer_days,0.88
1,price,freight_ratio,-0.78
2,carrier_to_customer_days,distance_km,0.64
3,days_vs_estimate,promised_days,-0.64
4,n_products,n_sellers,0.63
5,promised_days,distance_km,0.62


**What we see.**
- **Money, weight and counts are heavily right-skewed** (skew 6–22). The maxima are real orders (price up to R$13,440, weight up to 184 kg). They are kept; a linear model would need `log1p`, a tree model does not.
- **Count features are nearly constant:** the 99th percentile is 3 items, 2 products, 2 sellers and 2 payment rows. Their signal comes from a small tail of orders.
- **One redundant pair:** `delivery_days` and `carrier_to_customer_days` (ρ = 0.88). Carrier transit is most of the delivery time, so permutation importance will be split between them. `price` and `freight_ratio` (ρ = −0.78) are related by construction.
- **Category has 74 levels, 11 of them with fewer than 30 orders.** Group the rare levels or use an encoding that tolerates them.

## 8 · Temporal train / validation / test split

Orders are split by purchase date: **train** Jan–Dec 2017, **validation** Jan–Mar 2018 (model selection, threshold and calibration), **test** Apr–Aug 2018 (touched once, for the final numbers). The checks: are the three sets comparable, do the same customers and sellers appear across them, and does any label only become known after the next period has started.

In [9]:
m["split"] = np.select([m.order_purchase_timestamp < VAL_START, m.order_purchase_timestamp < TEST_START],
                       ["train", "validation"], default="test")
tr, va, te = (m[m.split == k] for k in ["train", "validation", "test"])
assert tr.order_purchase_timestamp.max() < va.order_purchase_timestamp.min() <= va.order_purchase_timestamp.max() < te.order_purchase_timestamp.min()
show(pd.DataFrame({
    "first purchase": [p.order_purchase_timestamp.min().date() for p in (tr, va, te)],
    "last purchase": [p.order_purchase_timestamp.max().date() for p in (tr, va, te)],
    "orders": [len(p) for p in (tr, va, te)],
    "share of table": [len(p) / len(m) for p in (tr, va, te)],
    "negative rate": [p.negative.mean() for p in (tr, va, te)],
    "label_uncertain": [p.label_uncertain.mean() for p in (tr, va, te)],
    "has seller history rate": [p.seller_prior_neg_rate.notna().mean() for p in (tr, va, te)],
}, index=["train", "validation", "test"]).reset_index(names="split"),
     num=["orders"], pct=["share of table", "negative rate", "label_uncertain", "has seller history rate"])

# leakage checks: each evaluation set against everything that comes before it
trva = pd.concat([tr, va])
leak = []
for name, past, future, cutoff in [("validation vs train", tr, va, VAL_START), ("test vs train + validation", trva, te, TEST_START)]:
    new_sellers = ~future.main_seller.drop_duplicates().isin(past.main_seller)
    print(f"{name}: {new_sellers.sum():,} of {len(new_sellers):,} sellers are not in the earlier data")
    leak.append({
        "comparison": name,
        "orders whose customer is also in the earlier data": future.customer_unique_id.isin(past.customer_unique_id).mean(),
        "orders from a seller not in the earlier data": (~future.main_seller.isin(past.main_seller)).mean(),
        "orders in a category not in the earlier data": (~future.category.isin(past.category)).mean(),
        "earlier orders delivered on or after the cut-off": (past.order_delivered_customer_date >= cutoff).mean(),
        "earlier orders reviewed on or after the cut-off": (past.review_answer_timestamp >= cutoff).mean()})
show(pd.DataFrame(leak).set_index("comparison").T.reset_index(names="check"),
     pct=["validation vs train", "test vs train + validation"])

drift = pd.DataFrame({"train median": tr[NUM].median(), "validation median": va[NUM].median(), "test median": te[NUM].median(),
                      "KS train vs validation": [ks_2samp(tr[f].dropna(), va[f].dropna()).statistic for f in NUM],
                      "KS train vs test": [ks_2samp(tr[f].dropna(), te[f].dropna()).statistic for f in NUM]})
show(drift.sort_values("KS train vs test", ascending=False).reset_index(names="feature"), num=list(drift.columns), dec=2)

monthly = m.groupby(m.order_purchase_timestamp.dt.to_period("M").astype(str)).agg(
    orders=("negative", "size"), negative_rate=("negative", "mean"), split=("split", "first"))
show(monthly.reset_index(names="month"), num=["orders"], pct=["negative_rate"])

,split,first purchase,last purchase,orders,share of table,negative rate,label_uncertain,has seller history rate
0,train,2017-01-05,2017-12-31,"40,714",45.7%,9.2%,0.5%,80.8%
1,validation,2018-01-01,2018-03-31,"17,864",20.0%,10.5%,0.6%,89.5%
2,test,2018-04-01,2018-08-29,"30,604",34.3%,8.6%,0.1%,86.7%


validation vs train: 347 of 1,298 sellers are not in the earlier data


test vs train + validation: 886 of 2,035 sellers are not in the earlier data


comparison,check,validation vs train,test vs train + validation
0,orders whose customer is also in the earlier data,1.5%,1.9%
1,orders from a seller not in the earlier data,9.9%,20.6%
2,orders in a category not in the earlier data,0.0%,0.0%
3,earlier orders delivered on or after the cut-off,4.4%,4.0%
4,earlier orders reviewed on or after the cut-off,5.8%,5.0%


,feature,train median,validation median,test median,KS train vs validation,KS train vs test
0,seller_prior_neg_rate,0.11,0.13,0.14,0.13,0.22
1,promised_days,23.61,24.06,21.39,0.07,0.21
2,delivery_days,10.63,11.89,7.90,0.09,0.21
3,carrier_to_customer_days,7.16,8.13,5.54,0.10,0.20
4,freight,16.50,16.69,18.38,0.04,0.19
5,approval_hours,0.28,0.34,0.42,0.11,0.18
6,seller_prior_n,32,73,71,0.18,0.17
7,handoff_days,2.38,2.37,1.81,0.06,0.14
8,days_vs_estimate,-13,-12,-13,0.10,0.09
9,desc_len,579,618,642,0.04,0.07


,month,orders,negative_rate,split
0,2017-01,720,10.3%,train
1,2017-02,"1,601",9.6%,train
2,2017-03,"2,414",9.0%,train
3,2017-04,"2,143",9.8%,train
4,2017-05,"3,418",8.9%,train
5,2017-06,"3,019",9.3%,train
6,2017-07,"3,737",8.5%,train
7,2017-08,"4,044",8.1%,train
8,2017-09,"3,941",8.3%,train
9,2017-10,"4,267",9.3%,train


**What we see.**
- **Sizes and base rates:** train 40,714 orders (9.2% negative), validation 17,864 (10.5%), test 30,604 (8.6%). Validation is the hardest quarter and test the easiest, so a metric will not carry over unchanged from one to the other.
- **The test period is faster:** median delivery is 10.6 days in train, 11.9 in validation and 7.9 in test; the promised window shortens from about 24 to 21 days (KS train vs test 0.19–0.22). Validation looks like train on these features. Order composition (items, sellers, price) does not shift anywhere.
- **Customers barely overlap** (1.5% of validation orders, 1.9% of test orders), so a customer-grouped split is not needed.
- **Sellers are a cold-start problem:** 9.9% of validation orders and 20.6% of test orders come from sellers absent from the earlier data. Seller identity must not be a feature; the point-in-time history carries seller information instead.
- **Labels that arrive late:** 5.8% of train orders were reviewed on or after 1 Jan 2018, and 5.0% of train + validation orders on or after 1 Apr 2018. A model trained as of the cut-off would not have had those labels yet.

**Leakage checks, in one place.**

| Check | Result |
|---|---|
| Features exist at the prediction point | Yes: every feature is known at purchase or at delivery (dictionary in §10) |
| Review text, review dates used as features | No: label and label checks only |
| Seller history looks ahead | No: only reviews answered strictly before the purchase; an order's own review is excluded |
| Label written before the outcome | 177 orders reviewed before delivery, flagged in `label_uncertain` (§5) |
| Sets overlap in time | No: asserted, train ends before validation starts, validation before test |
| Same customer on both sides | 1.5% / 1.9% of orders |
| Statistics fitted on evaluation data | None are fitted here; encoders, imputers and scalers must be fitted on train only |

**Why it matters.** The split is sound. Three things carry into modelling: compare each metric with its own set's base rate, expect the validation score to be a pessimistic guide to test because the test period is faster, and for a strict backtest drop the earlier orders whose review arrived after the cut-off.

## 9 · Does the label definition matter?

The target counts 1–2★ as negative. The same single-feature ranking is recomputed with 3★ also counted as negative.

In [10]:
neg3 = m.review_score <= 3
print(f"Negative rate with 1-2 stars: {m.negative.mean():.1%} | with 1-3 stars: {neg3.mean():.1%}")
sens = pd.DataFrame({"AUC, negative = 1-2 stars": [auc_free(m.negative, m[f]) for f in NUM + BIN],
                     "AUC, negative = 1-3 stars": [auc_free(neg3, m[f]) for f in NUM + BIN]}, index=NUM + BIN)
sens["rank (1-2)"] = sens.iloc[:, 0].rank(ascending=False).astype(int)
sens["rank (1-3)"] = sens.iloc[:, 1].rank(ascending=False).astype(int)
show(sens.sort_values("rank (1-2)").reset_index(names="feature"), num=list(sens.columns[:2]))

Negative rate with 1-2 stars: 9.2% | with 1-3 stars: 17.3%


,feature,"AUC, negative = 1-2 stars","AUC, negative = 1-3 stars",rank (1-2),rank (1-3)
0,n_items,0.593,0.559,1,3
1,seller_prior_neg_rate,0.587,0.568,2,2
2,freight,0.581,0.557,3,4
3,delivery_days,0.564,0.574,4,1
4,handoff_days,0.554,0.550,5,6
5,n_products,0.547,0.531,6,8
6,price,0.546,0.521,7,12
7,promised_days,0.546,0.540,8,7
8,carrier_to_customer_days,0.541,0.553,9,5
9,weight_g,0.538,0.525,10,10


**What we see.** Counting 3★ as negative almost doubles the positive rate (9.2% → 17.3%) and lowers most single-feature AUCs. The same four features lead under both definitions (`n_items`, `seller_prior_neg_rate`, `freight`, `delivery_days`), though their order changes: delivery time moves from 4th to 1st. Order size is tied more closely to strong dissatisfaction, and waiting time to mild dissatisfaction.

**Why it matters.** The 1–2★ target is kept. The conclusions of `01` about which features matter do not depend on that choice.

## 10 · Save, with a data dictionary

"Known at" is the earliest moment the value exists. Everything marked *purchase* or *delivery* is available at the prediction point; everything marked *after review* is the label or a label check and must never be a model input.

In [11]:
KEYS = ["order_id", "customer_unique_id", "main_seller", "order_purchase_timestamp", "order_delivered_customer_date", "split"]
LABEL = ["review_score", "negative", "label_uncertain", "review_conflict", "review_id_shared"]
out = m[KEYS + FEATURES + LABEL].reset_index(drop=True)
assert out.order_id.is_unique
out.to_parquet(DATA / "modelling_orders.parquet", index=False)
print(f"Saved {len(out):,} rows x {out.shape[1]} columns to {DATA / 'modelling_orders.parquet'}")

DICTIONARY = {  # column: (role, known at, meaning)
    "order_id": ("key", "purchase", "Order identifier; one row per order"),
    "customer_unique_id": ("key", "purchase", "Person behind the order (not a feature)"),
    "main_seller": ("key", "purchase", "Seller of the most expensive item (not a feature)"),
    "order_purchase_timestamp": ("key", "purchase", "Purchase time; defines the split"),
    "order_delivered_customer_date": ("key", "delivery", "Delivery time = prediction point"),
    "split": ("key", "purchase", "train (2017), validation (Jan-Mar 2018) or test (Apr-Aug 2018)"),
    "delivery_days": ("feature", "delivery", "Days from purchase to delivery"),
    "days_vs_estimate": ("feature", "delivery", "Delivery date minus estimated date, in days (0 or negative here)"),
    "promised_days": ("feature", "purchase", "Days from purchase to the estimated delivery date"),
    "handoff_days": ("feature", "delivery", "Days from purchase to carrier pick-up"),
    "carrier_to_customer_days": ("feature", "delivery", "Days from carrier pick-up to delivery"),
    "approval_hours": ("feature", "delivery", "Hours from purchase to payment approval"),
    "n_items": ("feature", "purchase", "Item rows in the order"),
    "n_products": ("feature", "purchase", "Distinct products in the order"),
    "n_sellers": ("feature", "purchase", "Distinct sellers in the order"),
    "price": ("feature", "purchase", "Sum of item prices (BRL)"),
    "freight": ("feature", "purchase", "Sum of freight charges (BRL)"),
    "freight_ratio": ("feature", "purchase", "freight / price"),
    "distance_km": ("feature", "purchase", "Main seller to customer, between zip-prefix centroids"),
    "installments": ("feature", "purchase", "Largest number of instalments among the order's payments"),
    "n_pay": ("feature", "purchase", "Payment rows for the order"),
    "photos": ("feature", "purchase", "Mean number of product photos across items"),
    "desc_len": ("feature", "purchase", "Mean product description length across items"),
    "weight_g": ("feature", "purchase", "Total product weight (g)"),
    "seller_prior_neg_rate": ("feature", "purchase", "Share of 1-2 star reviews among the seller's reviews answered before this purchase (needs 5; highest across the order's sellers)"),
    "seller_prior_n": ("feature", "purchase", "Number of those earlier reviews (smallest across the order's sellers)"),
    "dispatch_late": ("feature", "delivery", "Carrier pick-up after the seller's shipping deadline"),
    "category": ("feature", "purchase", "Category of the most expensive item"),
    "customer_state": ("feature", "purchase", "Customer's state"),
    "payment_type": ("feature", "purchase", "Type of the largest payment"),
    "review_score": ("label", "after review", "Review score, 1-5"),
    "negative": ("label", "after review", "TARGET: review_score is 1 or 2"),
    "label_uncertain": ("label check", "after review", "Conflicting reviews, or review answered before purchase or delivery"),
    "review_conflict": ("label check", "after review", "Several reviews with different scores"),
    "review_id_shared": ("label check", "after review", "The review is attached to other orders too"),
}
assert list(DICTIONARY) == list(out.columns), "data dictionary is out of sync with the saved columns"
assert {c for c, v in DICTIONARY.items() if v[0] == "feature"} == set(FEATURES)
show(pd.DataFrame(DICTIONARY, index=["role", "known at", "meaning"]).T
     .assign(dtype=out.dtypes.astype(str), missing=out.isna().mean()).reset_index(names="column"), pct=["missing"])

Saved 89,182 rows x 35 columns to dataset\modelling_orders.parquet


,column,role,known at,meaning,dtype,missing
0,order_id,key,purchase,Order identifier; one row per order,str,0.0%
1,customer_unique_id,key,purchase,Person behind the order (not a feature),str,0.0%
2,main_seller,key,purchase,Seller of the most expensive item (not a feature),str,0.0%
3,order_purchase_timestamp,key,purchase,Purchase time; defines the split,datetime64[us],0.0%
4,order_delivered_customer_date,key,delivery,Delivery time = prediction point,datetime64[us],0.0%
5,split,key,purchase,"train (2017), validation (Jan-Mar 2018) or test (Apr-Aug 2018)",str,0.0%
6,delivery_days,feature,delivery,Days from purchase to delivery,float64,0.0%
7,days_vs_estimate,feature,delivery,"Delivery date minus estimated date, in days (0 or negative here)",int64,0.0%
8,promised_days,feature,purchase,Days from purchase to the estimated delivery date,float64,0.0%
9,handoff_days,feature,delivery,Days from purchase to carrier pick-up,float64,0.2%


## Summary of decisions

| Area | Decision |
|---|---|
| Grain | One row per `order_id`, asserted after every join and before saving |
| Population | Delivered, reviewed, bought Jan 2017 – Aug 2018, on time: 89,182 orders, 9.2% negative |
| Several reviews per order | Most recent answer kept; conflicting scores flagged in `review_conflict` |
| Uncertain labels | Kept and flagged in `label_uncertain` (352 orders); shared reviews flagged in `review_id_shared` |
| Impossible durations | Set to missing (164 hand-off, 23 carrier-to-customer) |
| Zero instalments / zero weight | Instalments set to 1; weight set to missing |
| Distance | Centroids from in-Brazil coordinates, median per zip prefix; missing for 0.5% |
| Category | Portuguese name when untranslated; `(missing)` when the product has none |
| Seller history | Reviews answered before the purchase, from all reviewed orders; at least 5 needed for a rate |
| Outliers | Kept; they are real orders |
| Split | `split` column, by purchase date: train Jan–Dec 2017, validation Jan–Mar 2018, test Apr–Aug 2018 |

### To carry into modelling
- Do not use `main_seller` or `customer_unique_id` as features; they are keys.
- `delivery_days` and `carrier_to_customer_days` are near-duplicates (ρ = 0.88).
- 9.9% of validation orders and 20.6% of test orders come from sellers unseen in the earlier data.
- Base rates differ: train 9.2%, validation 10.5%, test 8.6%.
- Fit encoders, imputers and scalers on train only; tune on validation; use test once.
- For a strict backtest, exclude earlier orders reviewed on or after the cut-off (5.8% for validation, 5.0% for test).